In [3]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
import tpqoa
from datetime import datetime, timezone, timedelta
import time
import pickle
import warnings
warnings.filterwarnings('ignore')

# ==================== DATEN VORBEREITUNG ====================

def prepare_training_data(data, lookback_periods=[5, 15, 30], min_move_pips=3):
    """
    Erweiterte Feature-Engineering für bessere Vorhersagen

    Args:
        data: DataFrame mit 'price' Spalte
        lookback_periods: Liste von Perioden für Rolling-Features
        min_move_pips: Minimale Bewegung in Pips für sinnvolle Labels
    """
    df = data.copy()

    # Basis: Returns
    df['returns'] = np.log(df['price'] / df['price'].shift(1))

    # === TREND FEATURES ===
    for period in lookback_periods:
        # Rolling Returns (Momentum)
        df[f'momentum_{period}'] = df['returns'].rolling(period).sum()

        # Rolling Volatilität
        df[f'volatility_{period}'] = df['returns'].rolling(period).std()

        # Preis relativ zu SMA (Mean-Reversion Signal)
        df[f'price_vs_sma_{period}'] = (df['price'] - df['price'].rolling(period).mean()) / df['price'].rolling(period).std()

    # === MIKROSTRUKTUR FEATURES ===
    # High-Low Range (Volatilität)
    df['hl_range'] = (df['price'].rolling(10).max() - df['price'].rolling(10).min()) / df['price']



    # === RSI-ÄHNLICHER INDIKATOR ===
    for period in [14]:
        delta = df['returns']
        gain = (delta.where(delta > 0, 0)).rolling(period).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(period).mean()
        rs = gain / (loss + 1e-10)
        df[f'rsi_{period}'] = 100 - (100 / (1 + rs))

    # === ZEIT FEATURES ===
    df['hour'] = df.index.hour
    df['is_eu_session'] = ((df['hour'] >= 7) & (df['hour'] < 16)).astype(int)
    df['is_us_session'] = ((df['hour'] >= 13) & (df['hour'] < 21)).astype(int)

    # === LABEL ENGINEERING ===
    # Zukünftiger Return über nächste X Perioden
    forward_periods = 12  # ~1 Minute bei 5-Sekunden-Daten
    df['forward_return'] = df['price'].shift(-forward_periods) / df['price'] - 1

    # Nur labeln wenn Bewegung > min_move_pips
    min_move = min_move_pips * 0.0001  # EUR/USD: 1 pip = 0.0001

    df['direction'] = 0  # Neutral
    df.loc[df['forward_return'] > min_move, 'direction'] = 1   # Long
    df.loc[df['forward_return'] < -min_move, 'direction'] = -1  # Short

    # Nur Trades wenn Signal stark genug
    df['trade_signal'] = df['direction']

    df.dropna(inplace=True)

    return df


def train_model(data_path, output_model_path, output_params_path):
    """Trainiere verbessertes Modell"""

    # Daten laden
    data = pd.read_csv(data_path, parse_dates=['time'], index_col='time')
    data.rename(columns={'c': 'price'}, inplace=True, errors='ignore')

    # Features erstellen
    df = prepare_training_data(data, lookback_periods=[5, 15, 30], min_move_pips=3)

    # Feature-Spalten identifizieren
    feature_cols = [col for col in df.columns if col not in ['price', 'returns', 'forward_return', 'direction', 'trade_signal']]

    # Nur Daten mit Trade-Signalen (nicht neutral)
    train_df = df[df['direction'] != 0].copy()

    print(f"Training samples: {len(train_df)}")
    print(f"Class distribution:\n{train_df['direction'].value_counts()}")

    # Normalisierung
    means = train_df[feature_cols].mean()
    stds = train_df[feature_cols].std()
    train_df[feature_cols] = (train_df[feature_cols] - means) / (stds + 1e-10)

    # Random Forest statt LogReg (fängt Interaktionen besser ab)
    model = RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        min_samples_split=100,
        min_samples_leaf=50,
        class_weight='balanced',  # Wichtig bei Ungleichgewicht
        random_state=42,
        n_jobs=-1
    )

    model.fit(train_df[feature_cols], train_df['direction'])

    # Evaluation auf Trainingsdaten
    train_df['pred'] = model.predict(train_df[feature_cols])
    hits = (train_df['direction'] == train_df['pred']).sum()
    hit_ratio = hits / len(train_df)
    print(f"\nTraining Hit Ratio: {hit_ratio:.3f}")

    # Feature Importance
    feature_importance = pd.DataFrame({
        'feature': feature_cols,
        'importance': model.feature_importances_
    }).sort_values('importance', ascending=False)
    print("\nTop 10 Features:")
    print(feature_importance.head(10))

    # Modell und Parameter speichern
    pickle.dump(model, open(output_model_path, 'wb'))
    params = {
        'means': means,
        'stds': stds,
        'feature_cols': feature_cols,
        'lookback_periods': [5, 15, 30],
        'min_move_pips': 3
    }
    pickle.dump(params, open(output_params_path, 'wb'))

    print(f"\nModel saved to: {output_model_path}")
    print(f"Parameters saved to: {output_params_path}")

    return model, params


# ==================== TRADING BOT ====================

class ImprovedMLTrader(tpqoa.tpqoa):
    def __init__(self, conf_file, instrument, bar_length, model, params, units, confidence_threshold=0.6):
        super().__init__(conf_file)
        self.instrument = instrument
        self.bar_length = pd.to_timedelta(bar_length)
        self.tick_data = pd.DataFrame()
        self.raw_data = None
        self.data = None
        self.last_bar = None
        self.units = units
        self.position = 0
        self.profits = []

        # ML Komponenten
        self.model = model
        self.means = params['means']
        self.stds = params['stds']
        self.feature_cols = params['feature_cols']
        self.lookback_periods = params['lookback_periods']
        self.confidence_threshold = confidence_threshold  # Nur traden, wenn Modell sicher ist

        print(f"Bot initialized with confidence threshold: {confidence_threshold}")

    def get_most_recent(self, days=5):
        """Hole historische Daten"""
        while True:
            time.sleep(2)
            now = datetime.now(timezone.utc).replace(tzinfo=None, microsecond=0)
            past = now - timedelta(days=days)

            df = self.get_history(
                instrument=self.instrument,
                start=past,
                end=now,
                granularity='S5',
                price='M',
                localize=False
            ).c.dropna().to_frame()

            df.rename(columns={'c': 'price'}, inplace=True)
            df = df.resample(self.bar_length, label='right').last().dropna().iloc[:-1]

            self.raw_data = df.copy()
            self.last_bar = self.raw_data.index[-1]

            if pd.to_datetime(datetime.now(timezone.utc)) - self.last_bar < self.bar_length:
                print(f"Loaded {len(self.raw_data)} historical bars")
                break

    def on_success(self, time, bid, ask):
        print(f'\rTicks: {self.ticks}', end='', flush=True)

        recent_tick = pd.to_datetime(time)
        df = pd.DataFrame({self.instrument: (ask + bid) / 2}, index=[recent_tick])
        self.tick_data = pd.concat([self.tick_data, df])

        if recent_tick - self.last_bar > self.bar_length:
            self.resample_and_join()
            self.define_strategy()
            self.execute_trades()

    def resample_and_join(self):
        """Resample Tick-Daten zu Bars"""
        resampled = self.tick_data.resample(self.bar_length, label='right').last().ffill().iloc[:-1]
        resampled.rename(columns={self.instrument: 'price'}, inplace=True)

        # Sicherstellen, dass raw_data auch 'price' heißt
        if 'price' not in self.raw_data.columns:
            self.raw_data.rename(columns={self.instrument: 'price'}, inplace=True)

        self.raw_data = pd.concat([self.raw_data, resampled])
        self.tick_data = self.tick_data.iloc[-1:]
        self.last_bar = self.raw_data.index[-1]

    def define_strategy(self):
        """Feature-Berechnung und Vorhersage"""
        df = self.raw_data.copy()

        # Sicherstellen, dass Spalte 'price' heißt
        if 'price' not in df.columns:
            df.rename(columns={self.instrument: 'price'}, inplace=True)

        # Features berechnen (wie im Training)
        df = prepare_training_data(
            df,
            lookback_periods=self.lookback_periods,
            min_move_pips=3
        )

        if len(df) == 0:
            print("Warning: Not enough data for feature calculation")
            self.data = pd.DataFrame()
            return

        # Normalisierung
        df[self.feature_cols] = (df[self.feature_cols] - self.means) / (self.stds + 1e-10)

        # Vorhersage mit Confidence
        probas = self.model.predict_proba(df[self.feature_cols])
        max_proba = probas.max(axis=1)
        predictions = self.model.predict(df[self.feature_cols])

        # Nur traden, wenn Modell sicher ist
        df['position'] = 0
        confident_mask = max_proba >= self.confidence_threshold
        df.loc[confident_mask, 'position'] = predictions[confident_mask]

        self.data = df.copy()

        # Debug output
        if len(df) > 0:
            last_pred = df['position'].iloc[-1]
            last_conf = max_proba[-1]
            print(f" [Pred: {last_pred}, Conf: {last_conf:.2f}]", end=' ', flush=True)

    def execute_trades(self):
        """Trade-Ausführung basierend auf Position"""
        if len(self.data) == 0:
            return

        target_position = self.data['position'].iloc[-1]

        if target_position == 1 and self.position <= 0:
            # Go Long
            units_to_trade = self.units - self.position * self.units
            order = self.create_order(self.instrument, units_to_trade, suppress=True, ret=True)
            self.report_trade(order, 'GOING LONG')
            self.position = 1

        elif target_position == -1 and self.position >= 0:
            # Go Short
            units_to_trade = -self.units - self.position * self.units
            order = self.create_order(self.instrument, units_to_trade, suppress=True, ret=True)
            self.report_trade(order, 'GOING SHORT')
            self.position = -1

        elif target_position == 0 and self.position != 0:
            # Go Neutral
            order = self.create_order(self.instrument, -self.position * self.units, suppress=True, ret=True)
            self.report_trade(order, 'GOING NEUTRAL')
            self.position = 0

    def report_trade(self, order, going):
        """Trade-Reporting"""
        time = order['time']
        units = order['units']
        price = order['price']
        pl = float(order.get('pl', 0))
        self.profits.append(pl)
        cumpl = sum(self.profits)

        print(f"\n{'='*100}")
        print(f"{time} | {going}")
        print(f"{time} | units={units} | price={price} | P&L={pl:.2f} | Cum P&L={cumpl:.2f}")
        print('='*100 + '\n')

    def safe_stream_data(self, minutes=30, chunk_size=100, max_retries=5):
        """Timeout-sicherer Stream mit MINUTEN-Timer"""
        import time

        start_time = time.time()
        end_time = start_time + (minutes * 60)
        total_ticks = 0
        retry_count = 0

        print(f"\n SAFE Stream | {minutes} MINUTEN | Ende: {time.strftime('%H:%M:%S', time.localtime(end_time))}")

        while time.time() < end_time and retry_count < max_retries:
            try:
                print(f"\n **Laufzeit**: {int((time.time()-start_time)/60)}/{minutes}min | Ticks: {total_ticks}", end=" ")
                self.stream_data(self.instrument, stop=chunk_size)
                total_ticks += self.ticks
                print(f"+{self.ticks} | Total: {total_ticks}")

                retry_count = 0
                time.sleep(1)

            except Exception as e:
                retry_count += 1
                print(f"{type(e).__name__}: {str(e)[:40]}...")
                wait_time = min(retry_count * 2, 10)
                print(f"{wait_time}s Pause (Versuch {retry_count}/{max_retries})")
                time.sleep(wait_time)

            except KeyboardInterrupt:
                print("\n Manuell gestoppt")
                break

        runtime = int((time.time() - start_time) / 60)
        print(f"\n ENDE nach {runtime}min | {total_ticks} Ticks")
        return total_ticks


# ==================== USAGE EXAMPLE ====================

if __name__ == '__main__':
    # 1. Training (einmalig ausführen)

    # model, params = train_model(
    #     data_path=r'C:\Users\bedla\Documents\Ausbildung_Informatik\1_Praktikum\Praktikum_Daten-_und_Prozessanalyse\energy_trading\PyCharm_Notebook\Trading\Data\20251217_five_minute.csv',
    #     output_model_path=r'C:\Users\bedla\Documents\Ausbildung_Informatik\1_Praktikum\Praktikum_Daten-_und_Prozessanalyse\energy_trading\PyCharm_Notebook\Trading\Data\20251218_trained_model.pkl',
    #     output_params_path=r'C:\Users\bedla\Documents\Ausbildung_Informatik\1_Praktikum\Praktikum_Daten-_und_Prozessanalyse\energy_trading\PyCharm_Notebook\Trading\Data\20251218_params.pkl'
    # )


    # 2. Trading

    model = pickle.load(open(r'C:\Users\bedla\Documents\Ausbildung_Informatik\1_Praktikum\Praktikum_Daten-_und_Prozessanalyse\energy_trading\PyCharm_Notebook\Trading\Data\20251218_trained_model.pkl', 'rb'))
    params = pickle.load(open(r'C:\Users\bedla\Documents\Ausbildung_Informatik\1_Praktikum\Praktikum_Daten-_und_Prozessanalyse\energy_trading\PyCharm_Notebook\Trading\Data\20251218_params.pkl', 'rb'))

    trader = ImprovedMLTrader(
        conf_file=r'C:\Users\bedla\Documents\Ausbildung_Informatik\1_Praktikum\Praktikum_Daten-_und_Prozessanalyse\energy_trading\PyCharm_Notebook\Trading\Data\oanda.cfg',
        instrument='EUR_USD',
        bar_length='5min',  # Längere Bars!
        model=model,
        params=params,
        units=10000,
        confidence_threshold=0.515  # Nur traden, wenn Modell >65% sicher und somit den Wert auf 0.65 legen
    )

    trader.get_most_recent(days=5)
    trader.safe_stream_data(minutes=20)

    # Final position schließen
    if trader.position != 0:
        close_order = trader.create_order(
            trader.instrument,
            -trader.position * trader.units,
            suppress=True,
            ret=True
        )
        trader.report_trade(close_order, 'GOING NEUTRAL')


Bot initialized with confidence threshold: 0.515
Loaded 576 historical bars

 SAFE Stream | 20 MINUTEN | Ende: 10:24:20

Ticks: 100t**: 0/20min | Ticks: 0  [Pred: 0, Conf: 0.50] +100 | Total: 100

Ticks: 100t**: 1/20min | Ticks: 100 +100 | Total: 200

Ticks: 100t**: 3/20min | Ticks: 200  [Pred: 0, Conf: 0.50] +100 | Total: 300

Ticks: 100t**: 5/20min | Ticks: 300 +100 | Total: 400

Ticks: 100t**: 8/20min | Ticks: 400 +100 | Total: 500

Ticks: 100t**: 9/20min | Ticks: 500  [Pred: 0, Conf: 0.51] +100 | Total: 600

Ticks: 73it**: 10/20min | Ticks: 600 V20Timeout: v20 REST request to https://stream-fxpra...
2s Pause (Versuch 1/5)

 **Laufzeit**: 12/20min | Ticks: 600 V20Timeout: v20 REST request to https://stream-fxpra...
4s Pause (Versuch 2/5)

Ticks: 100t**: 13/20min | Ticks: 600 +100 | Total: 700

Ticks: 31it**: 14/20min | Ticks: 700  [Pred: -1, Conf: 0.53] 
2026-01-05T09:20:02.035690832Z | GOING SHORT
2026-01-05T09:20:02.035690832Z | units=-10000.0 | price=1.16824 | P&L=0.00 | Cum P&L=

V20Timeout: v20 REST request to https://api-fxpractice.oanda.com:443/v3/accounts/101-011-37633556-002/orders has timed out (read)